# ISS Group 24 Modeling — OWLv2 Few-Shot Single-Object Localizer

**Architecture**: `google/owlv2-base-patch16-ensemble` (frozen) + Multi-View Aggregator (~3M params) + Existence Head (~5K params), with optional LoRA on the last 4 ViT blocks at Stage 2.3.

**Pipeline**:
1. **Phase 0** — zero-shot OWLv2 baseline (no training).
2. **Stage 1.1** — train aggregator + existence head only; backbone & detection heads frozen.
3. **Stage 1.2** — also fine-tune OWLv2 box / class heads.
4. **Stage 2.3** — also LoRA-fine-tune the last 4 vision transformer blocks.

Each stage runs `epochs × K=5 folds`. Every (epoch, fold) writes a checkpoint and an analytics JSON.

**Before running on Colab:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"**.
2. *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it via checkpoint detection.

In [ ]:
# Enable autoreload so subsequent edits to modeling/*.py are picked up
# without restarting the kernel. Idempotent — safe to re-run.
%load_ext autoreload
%autoreload 2

import os
import sys
import subprocess
from pathlib import Path

In [ ]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"
USE_GOOGLE_COLAB = False

In [ ]:
if USE_GOOGLE_COLAB:
    from google.colab import drive


    def mount_drive() -> str:
        """Mount Google Drive and return the iss_group_24 project root path."""
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError(
            "Shared folder 'iss_group_24' not found on this Drive.\n"
            f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
        )

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = "."

In [ ]:
def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab. Torch / torchvision come pre-installed."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy",
            "transformers>=4.40", "accelerate>=0.30",
            "peft>=0.10", "huggingface_hub>=0.23",
            "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")

In [ ]:
MANIFEST  = f"{DRIVE_ROOT}/dataset/aggregated/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/aggregated"
OUT_ROOT  = f"{DRIVE_ROOT}/checkpoints"
ANALYSIS_ROOT = f"{DRIVE_ROOT}/analysis"

# Set CWD to the project root so relative paths in train.py land here.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
if USE_GOOGLE_COLAB:
    setup_repo()
    install_deps()

---
## Step 0 — Build Dataset Manifest (80/20 train/test + phase0)

Stages HOTS + InsDet into `dataset/aggregated/{train,test}` and vizwiz_novel into `dataset/aggregated/phase0`.  **Run once.** Idempotent — re-runs only when the staging directory is missing.

In [ ]:
import aggregator

train_dir = Path(DATA_ROOT) / 'train'
test_dir  = Path(DATA_ROOT) / 'test'
phase0_dir = Path(DATA_ROOT) / 'phase0'
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir() and any(train_dir.iterdir())
    and test_dir.is_dir() and any(test_dir.iterdir())
    and phase0_dir.is_dir() and any(phase0_dir.iterdir())
)
if staged_ok:
    print(f'Dataset already staged at {DATA_ROOT} — skipping aggregator.')
else:
    print('Running aggregator…')
    aggregator.main()

---
## Imports & shared kwargs

Every hyperparameter is here. Each per-stage cell pulls from `SHARED_KWARGS`; override with cell-local kwargs.

Defaults assume a 6 GB laptop GPU (RTX 2060): `img_size=768, batch_size=1, grad_accum_steps=8`. Bump these on Colab T4 (`img_size=960, batch_size=4, grad_accum_steps=2`).

In [ ]:
import torch
from modeling.train import (
    train_phase0,
    train_stage_1_1,
    train_stage_1_2,
    train_stage_2_3,
    evaluate_phase0,
    evaluate_run,
)
from modeling.plot import plot_all_from_jsons

_HAS_BIG_GPU = (
    torch.cuda.is_available()
    and torch.cuda.get_device_properties(0).total_memory > 10e9
)
_DEFAULT_IMG_SIZE = 960 if _HAS_BIG_GPU else 768
_DEFAULT_BATCH    = 4 if _HAS_BIG_GPU else 1
_DEFAULT_ACCUM    = 2 if _HAS_BIG_GPU else 8
print(f'Detected {"large" if _HAS_BIG_GPU else "small"} GPU → '
      f'img_size={_DEFAULT_IMG_SIZE}, batch={_DEFAULT_BATCH}, accum={_DEFAULT_ACCUM}')

In [ ]:
SHARED_KWARGS = dict(
    manifest        = MANIFEST,
    data_root       = DATA_ROOT,
    out_root        = OUT_ROOT,
    analysis_root   = ANALYSIS_ROOT,

    # Hardware
    img_size         = _DEFAULT_IMG_SIZE,
    batch_size       = _DEFAULT_BATCH,
    grad_accum_steps = _DEFAULT_ACCUM,
    num_workers      = 2,
    use_amp          = True,

    # Folds (every stage runs epochs × K folds; user contract).
    folds            = 5,
    fold_seed        = 42,
    n_support        = 4,
    neg_prob         = 0.5,

    # Stage durations (epochs are *per fold*; total passes = epochs × K).
    stage_1_1_epochs = 8,
    stage_1_2_epochs = 15,
    stage_2_3_epochs = 8,
    episodes_per_fold_s1 = 200,
    episodes_per_fold_s2 = 250,
    episodes_per_fold_s3 = 250,
    val_episodes     = 100,

    # LRs (per stage).
    lr_aggregator_s1 = 5e-4,
    lr_existence_s1  = 5e-4,
    lr_aggregator_s2 = 1e-4,
    lr_existence_s2  = 2e-4,
    lr_box_s2        = 5e-5,
    lr_class_s2      = 5e-5,
    lr_aggregator_s3 = 5e-5,
    lr_existence_s3  = 5e-5,
    lr_box_s3        = 2e-5,
    lr_class_s3      = 2e-5,
    lr_lora_s3       = 1e-4,
    weight_decay     = 1e-4,
    grad_clip        = 1.0,
    warmup_frac      = 0.05,

    # LoRA (Stage 2.3).
    lora_r           = 8,
    lora_alpha       = 16,
    lora_dropout     = 0.1,
    lora_layers      = 4,

    # Loss.
    focal_alpha      = 0.25,
    focal_gamma      = 2.0,
    lambda_l1        = 5.0,
    lambda_giou      = 2.0,
    anti_collapse_weight   = 0.1,
    box_size_threshold     = 0.6,
    existence_kl_threshold = 0.85,

    # Early stopping.
    early_stop_patience_s1 = 4,
    early_stop_patience_s2 = 5,
    early_stop_patience_s3 = 4,

    # I/O.
    keep_last_n      = 6,
    seed             = 42,
    device           = None,    # auto cuda / cpu
)

EVAL_KWARGS = dict(
    manifest    = MANIFEST,
    data_root   = DATA_ROOT,
    out_root    = OUT_ROOT,
    img_size    = _DEFAULT_IMG_SIZE,
    batch_size  = _DEFAULT_BATCH,
    num_workers = 2,
    n_support   = 4,
    neg_prob    = 0.5,
    test_episodes = 400,
    seed        = 42,

    # Tile inference (eval-only, can be turned off / on independently of training).
    # mode:
    #   'off'         — single-pass eval (cheapest)
    #   'pyramid_a'   — fixed 1+2x2 pyramid with 30% overlap (default; recommended)
    #   'hybrid_d'    — pyramid_a then re-tile the highest-scoring tile into 2x2
    # for_sources controls which dataset sources actually use tiling.  HOTS
    # scenes are small enough to skip tiling by default.
    tile_cfg = dict(
        mode                = 'pyramid_a',
        levels              = (1, 2),
        overlap             = 0.30,
        for_sources         = ('insdet',),
        nms_iou             = 0.5,
        top_k               = 100,
        score_combo         = 'existence_x_score',
        edge_score_penalty  = 0.5,
        edge_px             = 4,
        merge_partial_boxes = True,
        merge_min_score     = 0.2,
    ),
)

---
## Phase 0 — Zero-shot OWLv2 baseline (no training)

**Goal**: Establish the floor. OWLv2 is pre-trained on COCO + Objects365, so it already knows everyday indoor objects. We run image-guided detection: each of the 4 supports independently → average per-patch logits → pick top-1 box.

Runs on:
- `vizwiz_novel` (16 instances, 1 image each, rotation-synthesised supports)
- `test` split of HOTS + InsDet

**Decision gate**: if mAP@50 < 5% on **both** datasets, abort and inspect annotations.

In [ ]:
# Phase 0 — zero-shot OWLv2 baseline.
train_phase0(**SHARED_KWARGS)

### Evaluate Phase 0 on the test split

Re-runs the zero-shot baseline on the held-out test split only (HOTS + InsDet).

In [ ]:
evaluate_phase0(**{**EVAL_KWARGS, 'tile_cfg': {'mode': 'off'}})

---
## Stage 1.1 — Aggregator + Existence Head Warmup

**Goal**: Train the multi-view aggregator and existence head while keeping OWLv2 entirely frozen.

- **Trainable**: aggregator (~3M params), existence head (~5K params).
- **Frozen**: OWLv2 vision_model, class_head, box_head.
- **Loss**: focal existence loss + small existence-mean KL anti-collapse.
- **Per-stage layout**: `epochs × K=5 folds`, every (epoch, fold) writes `ckpt_foldF_epochE.pt` + `analysis/stage_1_1/epoch_E/fold_F.json`.

In [ ]:
train_stage_1_1(**SHARED_KWARGS)

### Evaluate Stage 1.1 on the test split

In [ ]:
# Tiling is on by default for InsDet (see EVAL_KWARGS['tile_cfg']).
# To disable for this run only: replace **EVAL_KWARGS with **{**EVAL_KWARGS, 'tile_cfg': {'mode': 'off'}}
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_1_1/best.pt',
    **EVAL_KWARGS,
)

---
## Stage 1.2 — Detection Head Fine-Tuning

**Goal**: Adapt OWLv2's `class_head` and `box_head` to our domain and instance-matching task. This is where most of the mAP gain happens.

- **Trainable**: aggregator, existence head, OWLv2 class_head, OWLv2 box_head, OWLv2 layer_norm.
- **Frozen**: OWLv2 vision_model.
- **Loss**: focal + L1 + GIoU + anti-collapse.
- **Box-head warmup**: the box head is frozen for the first 3 epochs so the existence head can calibrate before bbox loss fires.

In [ ]:
train_stage_1_2(
    **SHARED_KWARGS,
    stage_1_2_box_freeze_epochs=3,
)

### Evaluate Stage 1.2 on the test split

In [ ]:
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_1_2/best.pt',
    **EVAL_KWARGS,
)

---
## Stage 2.3 — LoRA Fine-Tuning of last 4 ViT blocks

**Goal**: Inject low-rank adapters into the q/v projections of OWLv2's last 4 vision transformer blocks. The frozen ViT acts as a regulariser; the backbone adapts slightly without forgetting general visual structure.

- **Trainable**: aggregator, existence head, class_head, box_head, LoRA adapters (~98K params).
- **Frozen**: rest of OWLv2 vision_model.
- **Loss**: same as Stage 1.2.

In [ ]:
train_stage_2_3(**SHARED_KWARGS)

### Evaluate Stage 2.3 on the test split

In [ ]:
evaluate_run(
    checkpoint=f'{OUT_ROOT}/stage_2_3/best.pt',
    **EVAL_KWARGS,
)

---
## Generate the offline report

Reads every per-(epoch, fold) JSON + aggregates + Phase 0 results, writes plot PNGs to `analysis/plots/`.

In [ ]:
plot_all_from_jsons(ANALYSIS_ROOT)

---
## Optional: export the trained model to ONNX / TFLite

Use the root-level `export.py` script.  ONNX is the primary target; TFLite is best-effort via `onnx2tf` (the OWLv2 attention stack does not have a clean direct TFLite path).  LoRA adapters are merged into the base weights before tracing.

In [ ]:
# Export the best Stage 2.3 checkpoint to ONNX.  Uncomment to run.
# !uv run python export.py \
#     --checkpoint checkpoints/stage_2_3/best.pt \
#     --out-dir exports/ \
#     --img-size {_DEFAULT_IMG_SIZE} \
#     --formats onnx

---
## Optional: ad-hoc inference on your own images

`inference.py::run_inference` takes 4 support paths + 1 query path and writes the originals + an annotated result image (with bbox + EXISTS/NOT EXISTS caption) to `inference/<NNNN>/`.  The tile mode is fully configurable per-call; pass `tile_cfg={'mode': 'off'}` to disable tiling, `'pyramid_a'` (default) for the recommended 1+2x2 pyramid, or `'hybrid_d'` for the recursive variant.

In [ ]:
# Example.  Replace the support / query paths with your own files.
# from inference import run_inference
#
# run_inference(
#     checkpoint=f'{OUT_ROOT}/stage_2_3/best.pt',
#     support_paths=['supports/s1.jpg', 'supports/s2.jpg', 'supports/s3.jpg', 'supports/s4.jpg'],
#     query_path='query/scene.jpg',
#     img_size=_DEFAULT_IMG_SIZE,
#     tile_cfg={'mode': 'pyramid_a'},   # or 'off' / 'hybrid_d'
# )